# Llama 3.2 1B — SRD → AXM → GGUF + Jetson Benchmark Suite

**Pipeline (run on Colab / RunPod A100):**
1. Download Llama 3.2-1B safetensors
2. SRD-4 fake-quantize → signed `.axm` governance container
3. Extract `.axm` → GGUF Q4_K_M for llama.cpp
4. Generate three Jetson benchmark scripts
5. SCP GGUF + scripts to Jetson Orin Nano

**Three benchmarks (run ON the Jetson):**

| # | Name | What it tests |
|---|------|---------------|
| 1 | Cognitive Shift Stress | 3K tokens predictable story → sudden hard logic puzzle; measures first-token latency after pivot |
| 2 | Long-Context Breadcrumb Memory | Unique facts planted every ~800 tokens; recall tested at 2K / 4K / 8K context depth |
| 3 | Power + Memory Under Reasoning | Python debug + chained riddles while `tegrastats` captures mW and EMC bandwidth |

**Jetson prerequisites** (run `setup_jetson.sh` once):
```
git clone https://github.com/ggerganov/llama.cpp /opt/llama.cpp
cd /opt/llama.cpp && mkdir build && cd build
cmake .. -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=87
cmake --build . -j4 -t llama-cli llama-bench
```

In [ ]:
# Cell 1 — pip install
import subprocess, sys
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.45", "accelerate", "bitsandbytes",
    "huggingface_hub", "safetensors", "sentencepiece", "tqdm",
], check=True)
print("✓ packages ready")

In [ ]:
# Cell 2 — clone axiom + AXIOM_MASTER_KEY
import os, secrets, subprocess, sys
from pathlib import Path

# ── Paths (edit for RunPod: /workspace/axiom) ──────────────────────────────
AXIOM_DIR  = Path("/content/axiom")       # RunPod: Path("/workspace/axiom")
OUTPUT_DIR = Path("/content/llama32_out") # RunPod: Path("/workspace/llama32_out")
BRANCH     = "claude/srd-prototype-benchmark-JRtv1"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "bench_scripts").mkdir(exist_ok=True)

if not AXIOM_DIR.is_dir():
    subprocess.run([
        "git", "clone", "--branch", BRANCH, "--depth", "1",
        "https://github.com/orivael-dev/axiom.git", str(AXIOM_DIR),
    ], check=True)
    print("✓ axiom cloned")
else:
    print(f"✓ axiom at {AXIOM_DIR}")
sys.path.insert(0, str(AXIOM_DIR))

# ── AXIOM_MASTER_KEY (persist across re-runs) ───────────────────────────────
KEY_FILE = OUTPUT_DIR / "axiom_master.key"
if os.environ.get("AXIOM_MASTER_KEY"):
    print("AXIOM_MASTER_KEY: from environment")
elif KEY_FILE.is_file():
    os.environ["AXIOM_MASTER_KEY"] = KEY_FILE.read_text().strip()
    print(f"AXIOM_MASTER_KEY: restored from {KEY_FILE}")
else:
    key = secrets.token_hex(32)
    os.environ["AXIOM_MASTER_KEY"] = key
    KEY_FILE.write_text(key)
    print(f"AXIOM_MASTER_KEY: generated → {KEY_FILE}")
    print("  ⚠ back this up — it's needed to verify the .axm later")

In [ ]:
# Cell 3 — ⚙️  CONFIGURE
import os

# ── Model ───────────────────────────────────────────────────────────────────
# Official gated model (requires HF_TOKEN + license acceptance):
#   MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
# Ungated community mirror (no token needed):
MODEL_ID = "unsloth/Llama-3.2-1B-Instruct"

HF_TOKEN = os.environ.get("HF_TOKEN", "")   # paste 'hf_...' for gated model

# ── SRD quantization ────────────────────────────────────────────────────────
SRD_TOP_K_PCT = 0.25   # 0.25 ≈ 4.5 bpw; 0.50 ≈ 6 bpw
GROUP_SIZE    = 64

# ── GGUF format ─────────────────────────────────────────────────────────────
GGUF_QUANT = "Q4_K_M"  # Q4_K_M / Q5_K_M / Q8_0

# ── Jetson SCP target ───────────────────────────────────────────────────────
JETSON_USER = "orin"            # ← edit
JETSON_HOST = "192.168.1.100"   # ← edit (or hostname)
JETSON_DEST = "/mnt/nvme/llama32"  # destination dir on Jetson (NVMe recommended)
JETSON_LLAMACPP = "/opt/llama.cpp/build/bin"  # where llama-cli lives on Jetson

# ── llama.cpp (for AXM → GGUF conversion, needed on THIS machine) ───────────
LLAMACPP_DIR = OUTPUT_DIR / "llama.cpp"

# Derived paths
SHORT_NAME  = MODEL_ID.split("/")[-1].lower().replace(".", "-")
AXM_PATH    = OUTPUT_DIR / f"{SHORT_NAME}_srd.axm"
GGUF_PATH   = OUTPUT_DIR / f"{SHORT_NAME}_srd_{GGUF_QUANT.lower()}.gguf"
STATS_JSON  = OUTPUT_DIR / f"{SHORT_NAME}_pack_stats.json"

print(f"Model:    {MODEL_ID}")
print(f"SRD:      top_k={SRD_TOP_K_PCT}  group={GROUP_SIZE}")
print(f"GGUF:     {GGUF_QUANT}")
print(f"AXM out:  {AXM_PATH}")
print(f"GGUF out: {GGUF_PATH}")
print(f"Jetson:   {JETSON_USER}@{JETSON_HOST}:{JETSON_DEST}")
if not HF_TOKEN:
    print("\n⚠ HF_TOKEN not set — using ungated mirror; set for meta-llama/ models")

In [ ]:
# Cell 4 — Download prebuilt llama.cpp (for AXM → GGUF conversion on THIS machine)
import json, platform, re, subprocess, sys, urllib.request
from pathlib import Path

LLAMACPP_DIR.mkdir(parents=True, exist_ok=True)
CLI = LLAMACPP_DIR / "llama-cli"
QUANTIZE_BIN = LLAMACPP_DIR / "llama-quantize"

def _find_cuda_version():
    try:
        out = subprocess.check_output(["nvcc", "--version"], stderr=subprocess.DEVNULL).decode()
        m = re.search(r"release (\d+\.\d+)", out)
        if m:
            return m.group(1).replace(".", "")
    except Exception:
        pass
    return "121"  # default CUDA 12.1

if CLI.exists() and QUANTIZE_BIN.exists():
    print(f"✓ llama-cli already at {CLI}")
else:
    cuda_ver = _find_cuda_version()
    print(f"Detected CUDA: {cuda_ver[:2]}.{cuda_ver[2:]}")

    # Find latest release with CUDA prebuilt asset
    try:
        releases_url = "https://api.github.com/repos/ggerganov/llama.cpp/releases"
        with urllib.request.urlopen(releases_url) as r:
            releases = json.loads(r.read())

        asset_url = None
        for release in releases[:5]:
            for asset in release.get("assets", []):
                name = asset["name"]
                if f"cuda{cuda_ver}" in name and "linux" in name.lower() and name.endswith(".tar.gz"):
                    asset_url = asset["browser_download_url"]
                    print(f"Found: {name}")
                    break
            if asset_url:
                break

        if asset_url:
            tar_path = LLAMACPP_DIR / "llama.tar.gz"
            print(f"Downloading prebuilt binary...")
            subprocess.run(["wget", "-q", "-O", str(tar_path), asset_url], check=True)
            subprocess.run(["tar", "-xzf", str(tar_path), "-C", str(LLAMACPP_DIR)], check=True)
            # Move binaries to top level
            for p in LLAMACPP_DIR.rglob("llama-cli"):
                p.rename(CLI)
                break
            for p in LLAMACPP_DIR.rglob("llama-quantize"):
                p.rename(QUANTIZE_BIN)
                break
            tar_path.unlink(missing_ok=True)
            print("✓ prebuilt llama.cpp binary installed")
        else:
            raise RuntimeError("No matching CUDA prebuilt asset found")

    except Exception as e:
        print(f"Prebuilt download failed ({e}), building from source...")
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/ggerganov/llama.cpp", str(LLAMACPP_DIR),
        ], check=True)
        build_dir = LLAMACPP_DIR / "build"
        build_dir.mkdir(exist_ok=True)
        subprocess.run(["cmake", "..", "-DGGML_CUDA=ON"], cwd=build_dir, check=True)
        subprocess.run(["cmake", "--build", ".", "-j4", "-t",
                        "llama-cli", "llama-quantize"], cwd=build_dir, check=True)
        CLI = build_dir / "bin" / "llama-cli"
        QUANTIZE_BIN = build_dir / "bin" / "llama-quantize"

# Also find convert_hf_to_gguf.py
CONVERT_SCRIPT = None
for p in LLAMACPP_DIR.rglob("convert_hf_to_gguf.py"):
    CONVERT_SCRIPT = p
    break
if CONVERT_SCRIPT is None:
    for p in LLAMACPP_DIR.rglob("convert-hf-to-gguf.py"):
        CONVERT_SCRIPT = p
        break

print(f"llama-cli:    {CLI}")
print(f"llama-quant:  {QUANTIZE_BIN}")
print(f"convert HF:   {CONVERT_SCRIPT}")

In [ ]:
# Cell 5 — SRD quantize → pack into signed .axm
import json, subprocess, sys, time
from pathlib import Path

PACK_SCRIPT = AXIOM_DIR / "research" / "quant" / "pack_to_axm.py"

if AXM_PATH.exists():
    gb = AXM_PATH.stat().st_size / 1024**3
    print(f"✓ AXM already packed: {AXM_PATH}  ({gb:.3f} GB)")
    print("  Delete the file and re-run to repack.")
else:
    print(f"SRD packing {MODEL_ID}")
    print(f"  top_k_pct={SRD_TOP_K_PCT}  group_size={GROUP_SIZE}")
    print(f"  output → {AXM_PATH}")
    print()

    cmd = [
        sys.executable, str(PACK_SCRIPT),
        "--model",      MODEL_ID,
        "--output",     str(AXM_PATH),
        "--top-k-pct",  str(SRD_TOP_K_PCT),
        "--group-size", str(GROUP_SIZE),
        "--stats-json", str(STATS_JSON),
    ]
    if HF_TOKEN:
        env_extra = {"HF_TOKEN": HF_TOKEN}
    else:
        env_extra = {}

    import os
    env = os.environ.copy()
    env.update(env_extra)

    t0 = time.time()
    result = subprocess.run(
        cmd, cwd=str(AXIOM_DIR), env=env,
        capture_output=False,   # stream output live
    )
    elapsed = time.time() - t0

    if result.returncode != 0:
        print(f"\n✗ Pack FAILED (rc={result.returncode})  {elapsed:.0f}s")
        raise RuntimeError("pack_to_axm.py failed — check output above")

    gb = AXM_PATH.stat().st_size / 1024**3
    print(f"\n✓ Done in {elapsed/60:.1f} min — {AXM_PATH.name}  ({gb:.3f} GB)")

if STATS_JSON.exists():
    stats = json.loads(STATS_JSON.read_text())
    print(f"  fingerprint: {stats.get('fingerprint')}")
    print(f"  bpw:         {stats.get('bpw_theoretical')}")
    print(f"  proofs:      {stats.get('proofs')}")

In [ ]:
# Cell 6 — Verify .axm proof chain
import json, subprocess, sys

AXM_CLI = AXIOM_DIR / "axm_cli.py"
out = subprocess.run(
    [sys.executable, str(AXM_CLI), "verify", str(AXM_PATH)],
    cwd=str(AXIOM_DIR), capture_output=True, text=True,
)
try:
    data = json.loads(out.stdout)
except Exception:
    data = {"verified": False, "error": out.stdout[-800:] + out.stderr[-400:]}

ok = data.get("verified", False)
print(f"{'✓ VERIFIED' if ok else '✗ FAILED'}")
print(f"  fingerprint:   {data.get('fingerprint', '?')}")
print(f"  proofs_checked: {data.get('proofs_checked', '?')}")
if not ok:
    print(f"  error: {data.get('error', 'unknown')}")
    raise RuntimeError("AXM verification failed — do not proceed")

In [ ]:
# Cell 7 — Extract .axm → GGUF Q4_K_M
import json, subprocess, sys, tempfile, time
from pathlib import Path

AXM_TO_GGUF = AXIOM_DIR / "research" / "quant" / "axm_to_gguf.py"

if GGUF_PATH.exists():
    gb = GGUF_PATH.stat().st_size / 1024**3
    print(f"✓ GGUF already exists: {GGUF_PATH}  ({gb:.3f} GB)")
else:
    print(f"Extracting AXM → GGUF {GGUF_QUANT}")
    print(f"  source:  {AXM_PATH}")
    print(f"  output:  {GGUF_PATH}")
    print(f"  llamacpp: {LLAMACPP_DIR}")

    t0 = time.time()
    result = subprocess.run(
        [
            sys.executable, str(AXM_TO_GGUF),
            "--container", str(AXM_PATH),
            "--gguf-out",  str(GGUF_PATH),
            "--llamacpp",  str(LLAMACPP_DIR),
            "--quant",     GGUF_QUANT,
        ],
        cwd=str(AXIOM_DIR),
        capture_output=False,
    )
    elapsed = time.time() - t0

    if result.returncode != 0:
        print(f"\n✗ GGUF extraction FAILED (rc={result.returncode})")
        raise RuntimeError("axm_to_gguf.py failed — check output above")

    gb = GGUF_PATH.stat().st_size / 1024**3
    print(f"\n✓ GGUF ready in {elapsed/60:.1f} min")
    print(f"  {GGUF_PATH.name}  ({gb:.3f} GB)")

In [ ]:
# Cell 8 — Local smoke test (runs on THIS machine, confirms GGUF works)
import re, subprocess, time

SMOKE_PROMPT = "In one sentence, what is machine learning?"

print(f"Smoke test: {CLI.name} -m {GGUF_PATH.name} -n 40")
print(f"Prompt: {SMOKE_PROMPT!r}")
print()

t0 = time.perf_counter()
r = subprocess.run(
    [
        str(CLI), "-m", str(GGUF_PATH),
        "--ngl", "99",
        "--ctx-size", "512",
        "--n-predict", "40",
        "--log-disable",
        "--prompt", SMOKE_PROMPT,
    ],
    capture_output=True, text=True, timeout=120,
)
elapsed = time.perf_counter() - t0

log = r.stdout + r.stderr
tps = None
for pattern in [
    r"(\d+\.\d+)\s+tokens per second",
    r"eval time.*?(\d+\.\d+)\s*tokens per second",
]:
    m = re.search(pattern, log)
    if m:
        tps = float(m.group(1))
        break

print(f"Status:   {'✓ OK' if r.returncode == 0 else '✗ FAILED'}")
print(f"Time:     {elapsed:.1f}s")
print(f"Tok/s:    {tps:.1f}" if tps else "Tok/s:    N/A")
print()
# Print generated text (skip echo of prompt)
stdout_clean = r.stdout.replace(SMOKE_PROMPT, "").strip()
if stdout_clean:
    print("Output:", stdout_clean[:300])
if r.returncode != 0:
    print("STDERR:", r.stderr[-300:])

In [ ]:
# Cell 9 — Generate Benchmark 1: Cognitive Shift Stress Test
from pathlib import Path

BENCH1 = OUTPUT_DIR / "bench_scripts" / "bench1_cognitive_shift.py"

BENCH1.write_text('''
"""Benchmark 1 — Cognitive Shift Stress Test (Jetson Orin Nano).

Phase 1: Feed the model ~3 000 tokens of a simple repeating bedtime story
         (maximally predictable, low-entropy text).
Phase 2: Without clearing context, pivot instantly to a hard multi-step
         logic puzzle — the kind that requires backtracking.

Measures
--------
- Phase 1 prefill time (how fast the 1B model absorbs predictable tokens)
- Time-to-first-token AFTER the pivot ("recovery time")
- Tokens/s during puzzle generation
- Whether the model ignores story context or gets confused by it

Usage
-----
    python3 bench1_cognitive_shift.py \\
        --gguf /mnt/nvme/llama32/llama-3-2-1b-instruct_srd_q4_k_m.gguf \\
        --llamacpp /opt/llama.cpp/build/bin
"""
from __future__ import annotations
import argparse, json, re, subprocess, sys, time
from pathlib import Path

# ── Story (Phase 1) ──────────────────────────────────────────────────────────
STORY_UNIT = (
    "Once upon a time in a sleepy forest, a little bear climbed over "
    "the mountain to see what she could see. She saw tall trees with golden "
    "leaves, a winding river full of silver fish, colorful birds singing in "
    "the branches, and wildflowers nodding in the gentle breeze. Satisfied "
    "with all she had seen, the little bear padded home through the soft pine "
    "needles to eat honey and sleep soundly under the stars. "
)
STORY_TOKENS_TARGET = 3000   # ~3K tokens of predictable context
story_repeats = max(1, STORY_TOKENS_TARGET // len(STORY_UNIT.split()))
PHASE1 = (STORY_UNIT * story_repeats).strip()

# ── Logic puzzle (Phase 2) ───────────────────────────────────────────────────
PIVOT = """

--- CONTEXT SHIFT ---
Ignore the story above. Solve this logic puzzle step by step:

Three logicians A, B, and C each wear a hat that is either red or blue.
Each can see the others\' hats but not their own.
A says: "I cannot determine my hat color."
B says: "I cannot determine my hat color."
C says: "I know my hat color. It is red."

What is the color of every hat? Reason through each step carefully.
"""

FULL_PROMPT = PHASE1 + PIVOT


def find_cli(llamacpp_bin: str) -> Path:
    for name in ("llama-cli", "main"):
        p = Path(llamacpp_bin) / name
        if p.exists():
            return p
    raise FileNotFoundError(f"llama-cli not found in {llamacpp_bin}")


def parse_tps(log: str) -> float | None:
    for pat in [r"(\\d+\\.\\d+)\\s+tokens per second",
                r"eval time.*?(\\d+\\.\\d+)\\s*tokens per second"]:
        m = re.search(pat, log)
        if m:
            return float(m.group(1))
    return None


def parse_timing(log: str, key: str) -> float | None:
    m = re.search(rf"{key}\\s*=\\s*([\\d.]+)\\s*ms", log, re.IGNORECASE)
    return float(m.group(1)) / 1000.0 if m else None


def run(args):
    cli = find_cli(args.llamacpp)
    ctx = len(FULL_PROMPT.split()) * 2   # rough upper bound
    ctx = min(max(ctx, 4096), args.max_ctx)
    prompt_tokens_est = len(PHASE1.split())

    print("=" * 64)
    print("BENCHMARK 1 — Cognitive Shift Stress Test")
    print("=" * 64)
    print(f"Model:         {args.gguf}")
    print(f"Phase 1 prompt: ~{prompt_tokens_est} tokens of predictable story")
    print(f"Phase 2 pivot:  logic puzzle (immediately after)")
    print(f"ctx-size:       {ctx}")
    print(f"ngl:            {args.ngl}")
    print()

    t0 = time.perf_counter()
    r = subprocess.run(
        [
            str(cli), "-m", args.gguf,
            "--ngl",       str(args.ngl),
            "--ctx-size",  str(ctx),
            "--n-predict", str(args.n_predict),
            "--log-disable",
            "--prompt",    FULL_PROMPT,
        ],
        capture_output=True, text=True, timeout=600,
    )
    elapsed = time.perf_counter() - t0
    log = r.stdout + r.stderr

    prefill_s   = parse_timing(log, "prompt eval time")
    load_s      = parse_timing(log, "load_time")
    tps         = parse_tps(log)
    ttft_s      = parse_timing(log, "eval time")   # first eval chunk

    # Extract generated puzzle response (everything after the pivot marker)
    response_text = ""
    if "--- CONTEXT SHIFT ---" in r.stdout:
        response_text = r.stdout.split("--- CONTEXT SHIFT ---")[-1][-800:]
    elif r.stdout:
        response_text = r.stdout[-600:]

    print(f"Status:       {'✓ OK' if r.returncode == 0 else '✗ FAILED'}")
    print(f"Total time:   {elapsed:.1f}s")
    print(f"Load time:    {load_s:.2f}s" if load_s else "Load time:    N/A")
    print(f"Prefill time: {prefill_s:.2f}s" if prefill_s else "Prefill time: N/A")
    print(f"Tok/s:        {tps:.2f}" if tps else "Tok/s:        N/A")
    print()
    print("--- Model response (puzzle section) ---")
    print(response_text.strip())

    results = {
        "benchmark": "cognitive_shift",
        "model": args.gguf,
        "status": "ok" if r.returncode == 0 else "failed",
        "elapsed_s": round(elapsed, 2),
        "load_s": round(load_s, 3) if load_s else None,
        "prefill_s": round(prefill_s, 3) if prefill_s else None,
        "tok_per_s": round(tps, 2) if tps else None,
        "phase1_tokens_est": prompt_tokens_est,
        "ctx": ctx,
        "ngl": args.ngl,
        "response_snippet": response_text.strip()[:400],
    }

    if args.json_out:
        Path(args.json_out).write_text(json.dumps(results, indent=2))
        print(f"\nResults → {args.json_out}")

    return results


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--gguf",       required=True)
    ap.add_argument("--llamacpp",   required=True)
    ap.add_argument("--ngl",        type=int, default=99)
    ap.add_argument("--max-ctx",    type=int, default=8192)
    ap.add_argument("--n-predict",  type=int, default=256,
                    help="tokens to generate for the puzzle answer")
    ap.add_argument("--json-out",   default=None)
    run(ap.parse_args())


if __name__ == "__main__":
    main()
''')

print(f"✓ Written: {BENCH1}")

In [ ]:
# Cell 10 — Generate Benchmark 2: Long-Context Breadcrumb Memory Test
from pathlib import Path

BENCH2 = OUTPUT_DIR / "bench_scripts" / "bench2_breadcrumb_memory.py"

BENCH2.write_text('''
"""Benchmark 2 — Long-Context Breadcrumb Memory Test (Jetson Orin Nano).

Plants five unique, specific facts (breadcrumbs) across a long conversation.
Each breadcrumb is surrounded by neutral filler text.
After N tokens of total context, asks the model to recall all five facts.

Tests three context depths:
  - 2 048 tokens  (short — should recall easily)
  - 4 096 tokens  (medium — tests real attention span)
  - 8 192 tokens  (long   — where 1B models often start forgetting)

Scores each run: how many of the 5 breadcrumbs were correctly recalled.

Usage
-----
    python3 bench2_breadcrumb_memory.py \\
        --gguf /mnt/nvme/llama32/model.gguf \\
        --llamacpp /opt/llama.cpp/build/bin
"""
from __future__ import annotations
import argparse, json, re, subprocess, sys, time
from pathlib import Path

# ── Breadcrumbs ──────────────────────────────────────────────────────────────
# Each is a (key, fact, recall_check) tuple.
# recall_check: substring that MUST appear in the model response for that fact.
BREADCRUMBS = [
    ("animal",   "My favorite imaginary animal is a neon green turtle named Sparky.",
                 ["sparky", "neon green", "turtle"]),
    ("food",     "The only food Sparky eats is glowing purple mushrooms from the moonlit cave.",
                 ["purple mushroom", "glowing", "mushroom"]),
    ("password", "The secret password to my treehouse fort is THUNDERCLOUD42.",
                 ["thundercloud42", "thundercloud"]),
    ("planet",   "My invisible best friend Commander Zox lives on planet Zorzog.",
                 ["zorzog", "zox"]),
    ("color",    "My lucky number is 7 and my lucky color is cobalt blue.",
                 ["cobalt blue", "cobalt", "lucky color"]),
]

FILLER_UNIT = (
    "We spent the afternoon exploring the garden and talking about many "
    "interesting things. The clouds were fluffy and the air smelled like "
    "fresh grass. We saw some butterflies and counted their wings. Then we "
    "had lemonade and talked about what we might do tomorrow. "
)

RECALL_PROMPT = """

--- RECALL TEST ---
Without looking back, please answer these questions:
1. What is the name of my favorite imaginary animal, and what color is it?
2. What is the only food that my favorite animal eats?
3. What is the secret password to my treehouse fort?
4. What is the name of my invisible best friend, and what planet do they live on?
5. What is my lucky number and my lucky color?
"""


def build_prompt(target_tokens: int) -> str:
    """Build a prompt with breadcrumbs spaced evenly through filler text."""
    filler_words = len(FILLER_UNIT.split())
    bc_words = sum(len(bc[1].split()) for bc in BREADCRUMBS)
    recall_words = len(RECALL_PROMPT.split())
    filler_needed = max(0, target_tokens - bc_words - recall_words)
    filler_repeats_each = max(1, filler_needed // (len(BREADCRUMBS) * filler_words))

    parts = []
    for _, fact, _ in BREADCRUMBS:
        parts.append(fact)
        parts.append((FILLER_UNIT * filler_repeats_each).strip())
    parts.append(RECALL_PROMPT)
    return " ".join(parts)


def find_cli(llamacpp_bin: str) -> Path:
    for name in ("llama-cli", "main"):
        p = Path(llamacpp_bin) / name
        if p.exists():
            return p
    raise FileNotFoundError(f"llama-cli not found in {llamacpp_bin}")


def score_response(response: str) -> dict:
    """Check how many breadcrumbs appear correctly in the response."""
    r = response.lower()
    hits, misses = [], []
    for key, _, checks in BREADCRUMBS:
        recalled = any(c.lower() in r for c in checks)
        (hits if recalled else misses).append(key)
    return {"score": len(hits), "total": len(BREADCRUMBS),
            "recalled": hits, "forgotten": misses}


def run_at_depth(cli, gguf, ctx_tokens, ngl, n_predict):
    prompt = build_prompt(ctx_tokens)
    actual_words = len(prompt.split())

    t0 = time.perf_counter()
    r = subprocess.run(
        [
            str(cli), "-m", gguf,
            "--ngl",       str(ngl),
            "--ctx-size",  str(ctx_tokens + n_predict + 64),
            "--n-predict", str(n_predict),
            "--log-disable",
            "--prompt",    prompt,
        ],
        capture_output=True, text=True, timeout=600,
    )
    elapsed = time.perf_counter() - t0

    log = r.stdout + r.stderr
    tps = None
    for pat in [r"(\\d+\\.\\d+)\\s+tokens per second",
                r"eval time.*?(\\d+\\.\\d+)\\s*tokens per second"]:
        m = re.search(pat, log)
        if m:
            tps = float(m.group(1))
            break

    # Extract the recall response (after RECALL TEST marker)
    response = ""
    if "--- RECALL TEST ---" in r.stdout:
        response = r.stdout.split("--- RECALL TEST ---")[-1]
    elif r.stdout:
        response = r.stdout[-1200:]

    sc = score_response(response)
    return {
        "ctx_target": ctx_tokens,
        "prompt_words": actual_words,
        "status": "ok" if r.returncode == 0 else "failed",
        "elapsed_s": round(elapsed, 2),
        "tok_per_s": round(tps, 2) if tps else None,
        "score": sc["score"],
        "total": sc["total"],
        "recalled": sc["recalled"],
        "forgotten": sc["forgotten"],
        "response_snippet": response.strip()[:600],
    }


def run(args):
    cli = find_cli(args.llamacpp)
    depths = [2048, 4096, 8192]
    all_results = {}

    print("=" * 64)
    print("BENCHMARK 2 — Long-Context Breadcrumb Memory Test")
    print("=" * 64)
    print(f"Model: {args.gguf}")
    print(f"ngl:   {args.ngl}")
    print(f"Breadcrumbs planted: {len(BREADCRUMBS)}")
    print()

    for depth in depths:
        print(f"--- Context depth: {depth} tokens ---")
        row = run_at_depth(cli, args.gguf, depth, args.ngl, args.n_predict)
        all_results[str(depth)] = row
        score_str = f"{row[\'score\']}/{row[\'total\']}  recalled={row[\'recalled\']}  forgotten={row[\'forgotten\']}"
        tps_str   = f"{row[\'tok_per_s\']:.2f} tok/s" if row.get(\'tok_per_s\') else "N/A tok/s"
        print(f"  Score: {score_str}")
        print(f"  Speed: {tps_str}  {row[\'elapsed_s\']:.1f}s")
        print(f"  Response:\n{row[\'response_snippet\'][:400]}")
        print()

    print("=" * 64)
    print("SUMMARY")
    print(f"  {\' ctx\':<8}  {\'score\':<8}  {\'tok/s\':<8}  {\'status\':<8}")
    for depth, r in all_results.items():
        tps_s = f"{r[\'tok_per_s\']:.1f}" if r.get(\'tok_per_s\') else "N/A"
        print(f"  {depth:<8}  {r[\'score\']}/{r[\'total\']:<6}  {tps_s:<8}  {r[\'status\']}")

    if args.json_out:
        out = {"benchmark": "breadcrumb_memory", "model": args.gguf,
               "ngl": args.ngl, "depths": all_results}
        Path(args.json_out).write_text(json.dumps(out, indent=2))
        print(f"\nResults → {args.json_out}")

    return all_results


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--gguf",      required=True)
    ap.add_argument("--llamacpp",  required=True)
    ap.add_argument("--ngl",       type=int, default=99)
    ap.add_argument("--n-predict", type=int, default=300)
    ap.add_argument("--json-out",  default=None)
    run(ap.parse_args())


if __name__ == "__main__":
    main()
''')

print(f"✓ Written: {BENCH2}")

In [ ]:
# Cell 11 — Generate Benchmark 3: Power + Memory Efficiency Under Reasoning
from pathlib import Path

BENCH3 = OUTPUT_DIR / "bench_scripts" / "bench3_power_reasoning.py"

BENCH3.write_text('''
"""Benchmark 3 — Power + Memory Efficiency Under Reasoning (Jetson Orin Nano).

Forces detailed nuanced reasoning while tegrastats captures:
  - Power draw per rail (mW): VDD_CPU_GPU_CV, VDD_SOC, total
  - EMC (External Memory Controller) frequency (MHz) → memory bandwidth proxy
  - RAM usage (MB)

Two reasoning tasks:
  Task A: Debug a multi-bug Python loop (syntax + logic + name errors)
  Task B: Solve a chain of three riddles with step-by-step reasoning

Tegrastats runs in the background during inference. Sampled every 500ms.
Peak power, average power, and peak EMC are reported.

Usage
-----
    python3 bench3_power_reasoning.py \\
        --gguf /mnt/nvme/llama32/model.gguf \\
        --llamacpp /opt/llama.cpp/build/bin

    # On Jetson only (tegrastats not available on x86)
    # If tegrastats is absent, power columns will show N/A.
"""
from __future__ import annotations
import argparse, json, os, re, subprocess, sys, tempfile, threading, time
from pathlib import Path

# ── Reasoning tasks ──────────────────────────────────────────────────────────

TASK_A_DEBUG = """\
You are an expert Python debugger. Find and fix every bug in this code.
List each bug clearly, explain why it is wrong, and provide corrected code.

```python
data = [10, 15, 20, 25, 30]
running_total = 0
results = []

for i in range(len(data)):
    if data[i] % 2 = 0:          # bug 1
        running_total =+ data[i]  # bug 2
        results.append(data[i] * 2
                                  # bug 3 (unclosed paren)

print("Even doubled:", result)   # bug 4 (wrong name)
print("Sum of evens:", running_total)
print("Average:", running_total / len(results))
```

After listing all bugs and corrections, write the complete corrected program.
"""

TASK_B_RIDDLES = """\
Solve all three riddles. For each, state your answer and show every reasoning step.

Riddle 1:
I speak without a mouth and hear without ears. I have no body, but I
come alive with the wind. What am I?

Riddle 2:
The more you take, the more you leave behind. What am I?

Riddle 3:
I have cities, but no houses live there. I have mountains, but no trees
grow there. I have water, but no fish swim there. I have roads, but no
cars drive there. What am I?

After answering all three, invent a fourth riddle that uses the same
structural pattern ("I have X but no Y") about a concept from computer science.
"""

TASKS = [
    {"name": "python_debug",   "prompt": TASK_A_DEBUG,   "n_predict": 512},
    {"name": "chain_riddles",  "prompt": TASK_B_RIDDLES,  "n_predict": 400},
]


# ── tegrastats reader ────────────────────────────────────────────────────────

class TegrastatsMonitor:
    """Launch tegrastats in background, collect power + EMC samples."""

    def __init__(self, interval_ms: int = 500):
        self.interval_ms = interval_ms
        self._samples: list[dict] = []
        self._proc = None
        self._thread = None
        self._running = False

    def start(self):
        try:
            self._proc = subprocess.Popen(
                ["tegrastats", "--interval", str(self.interval_ms)],
                stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True,
            )
            self._running = True
            self._thread = threading.Thread(target=self._collect, daemon=True)
            self._thread.start()
        except FileNotFoundError:
            print("  [tegrastats not found — power readings will be N/A]")

    def stop(self):
        self._running = False
        if self._proc:
            self._proc.terminate()
        if self._thread:
            self._thread.join(timeout=3)

    def _collect(self):
        for line in self._proc.stdout:
            if not self._running:
                break
            s = self._parse_line(line)
            if s:
                self._samples.append(s)

    def _parse_line(self, line: str) -> dict | None:
        sample = {}
        # RAM <used>/<total>MB
        m = re.search(r"RAM\\s+(\\d+)/(\\d+)MB", line)
        if m:
            sample["ram_used_mb"] = int(m.group(1))
            sample["ram_total_mb"] = int(m.group(2))
        # EMC_FREQ <N>%@<MHz>
        m = re.search(r"EMC_FREQ\\s+\\d+%@(\\d+)", line)
        if m:
            sample["emc_mhz"] = int(m.group(1))
        # VDD_CPU_GPU_CV <mW>/<mW>
        m = re.search(r"VDD_CPU_GPU_CV\\s+(\\d+)/(\\d+)", line)
        if m:
            sample["gpu_cpu_mw"] = int(m.group(1))
        # VDD_SOC <mW>/<mW>
        m = re.search(r"VDD_SOC\\s+(\\d+)/(\\d+)", line)
        if m:
            sample["soc_mw"] = int(m.group(1))
        # Sum for total
        total = sum(v for k, v in sample.items() if k.endswith("_mw"))
        if total:
            sample["total_mw"] = total
        return sample or None

    def summary(self) -> dict:
        if not self._samples:
            return {"available": False}
        def peak_avg(key):
            vals = [s[key] for s in self._samples if key in s]
            if not vals:
                return None, None
            return max(vals), round(sum(vals) / len(vals), 1)
        gpu_pk, gpu_avg = peak_avg("gpu_cpu_mw")
        soc_pk, soc_avg = peak_avg("soc_mw")
        tot_pk, tot_avg = peak_avg("total_mw")
        emc_pk, emc_avg = peak_avg("emc_mhz")
        ram_pk, _ = peak_avg("ram_used_mb")
        return {
            "available": True,
            "samples": len(self._samples),
            "gpu_cpu_mw": {"peak": gpu_pk, "avg": gpu_avg},
            "soc_mw":     {"peak": soc_pk, "avg": soc_avg},
            "total_mw":   {"peak": tot_pk, "avg": tot_avg},
            "emc_mhz":    {"peak": emc_pk, "avg": emc_avg},
            "ram_peak_mb": ram_pk,
        }


# ── llama runner ─────────────────────────────────────────────────────────────

def find_cli(llamacpp_bin: str) -> Path:
    for name in ("llama-cli", "main"):
        p = Path(llamacpp_bin) / name
        if p.exists():
            return p
    raise FileNotFoundError(f"llama-cli not found in {llamacpp_bin}")


def parse_tps(log: str) -> float | None:
    for pat in [r"(\\d+\\.\\d+)\\s+tokens per second",
                r"eval time.*?(\\d+\\.\\d+)\\s*tokens per second"]:
        m = re.search(pat, log)
        if m:
            return float(m.group(1))
    return None


def run_task(cli, gguf, task, ngl):
    monitor = TegrastatsMonitor(interval_ms=500)
    monitor.start()

    t0 = time.perf_counter()
    r = subprocess.run(
        [
            str(cli), "-m", gguf,
            "--ngl",       str(ngl),
            "--ctx-size",  "2048",
            "--n-predict", str(task["n_predict"]),
            "--log-disable",
            "--prompt",    task["prompt"],
        ],
        capture_output=True, text=True, timeout=600,
    )
    elapsed = time.perf_counter() - t0

    monitor.stop()
    power = monitor.summary()
    log = r.stdout + r.stderr
    tps = parse_tps(log)

    return {
        "task": task["name"],
        "status": "ok" if r.returncode == 0 else "failed",
        "elapsed_s": round(elapsed, 2),
        "tok_per_s": round(tps, 2) if tps else None,
        "power": power,
        "response_snippet": r.stdout.strip()[-600:],
    }


def run(args):
    cli = find_cli(args.llamacpp)
    all_results = {}

    print("=" * 64)
    print("BENCHMARK 3 — Power + Memory Under Reasoning")
    print("=" * 64)
    print(f"Model: {args.gguf}")
    print(f"ngl:   {args.ngl}")
    print()

    for task in TASKS:
        print(f"--- Task: {task[\'name\']} ---")
        result = run_task(cli, args.gguf, task, args.ngl)
        all_results[task["name"]] = result

        tps_s = f"{result[\'tok_per_s\']:.2f} tok/s" if result.get(\'tok_per_s\') else "N/A"
        pw = result["power"]
        if pw.get("available"):
            print(f"  Status:      {result[\'status\']}")
            print(f"  Tok/s:       {tps_s}  ({result[\'elapsed_s\']}s)")
            print(f"  CPU+GPU pwr: peak {pw[\'gpu_cpu_mw\'][\'peak\']} mW  avg {pw[\'gpu_cpu_mw\'][\'avg\']} mW")
            print(f"  SoC power:   peak {pw[\'soc_mw\'][\'peak\']} mW  avg {pw[\'soc_mw\'][\'avg\']} mW")
            print(f"  Total power: peak {pw[\'total_mw\'][\'peak\']} mW  avg {pw[\'total_mw\'][\'avg\']} mW")
            print(f"  EMC freq:    peak {pw[\'emc_mhz\'][\'peak\']} MHz  avg {pw[\'emc_mhz\'][\'avg\']} MHz")
            print(f"  RAM peak:    {pw[\'ram_peak_mb\']} MB")
        else:
            print(f"  Status: {result[\'status\']}  Tok/s: {tps_s}  (tegrastats N/A)")
        print(f"  Response snippet:\n{result[\'response_snippet\'][-400:]}")
        print()

    print("=" * 64)
    print("SUMMARY")
    print(f"  {\' task\':<20}  {\'tok/s\':<8}  {\'peak W\':<9}  {\'avg W\':<8}  {\'peak EMC\':<10}  status")
    for name, r in all_results.items():
        tps_s = f"{r[\'tok_per_s\']:.2f}" if r.get(\'tok_per_s\') else "N/A"
        pw = r["power"]
        if pw.get("available"):
            pk_w = f"{pw[\'total_mw\'][\'peak\']/1000:.2f}W" if pw[\'total_mw\'][\'peak\'] else "N/A"
            av_w = f"{pw[\'total_mw\'][\'avg\']/1000:.2f}W" if pw[\'total_mw\'][\'avg\'] else "N/A"
            em   = f"{pw[\'emc_mhz\'][\'peak\']} MHz" if pw[\'emc_mhz\'][\'peak\'] else "N/A"
        else:
            pk_w, av_w, em = "N/A", "N/A", "N/A"
        print(f"  {name:<20}  {tps_s:<8}  {pk_w:<9}  {av_w:<8}  {em:<10}  {r[\'status\']}") 

    if args.json_out:
        out = {"benchmark": "power_reasoning", "model": args.gguf,
               "ngl": args.ngl, "tasks": all_results}
        Path(args.json_out).write_text(json.dumps(out, indent=2))
        print(f"\nResults → {args.json_out}")

    return all_results


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--gguf",      required=True)
    ap.add_argument("--llamacpp",  required=True)
    ap.add_argument("--ngl",       type=int, default=99)
    ap.add_argument("--json-out",  default=None)
    run(ap.parse_args())


if __name__ == "__main__":
    main()
''')

print(f"✓ Written: {BENCH3}")

In [ ]:
# Cell 12 — Generate run_jetson_bench.sh (master script for Jetson)
from pathlib import Path

JETSON_GGUF_PATH = f"{JETSON_DEST}/{GGUF_PATH.name}"
MASTER_SH = OUTPUT_DIR / "bench_scripts" / "run_jetson_bench.sh"

MASTER_SH.write_text(f"""#!/bin/bash
# run_jetson_bench.sh — Run all three Llama 3.2 1B benchmarks on Jetson Orin Nano
#
# Prerequisites:
#   - llama.cpp built at {JETSON_LLAMACPP}
#   - GGUF at {JETSON_GGUF_PATH}
#   - This script and bench scripts in {JETSON_DEST}/bench_scripts/
#
# Run: bash {JETSON_DEST}/bench_scripts/run_jetson_bench.sh

set -e

GGUF="{JETSON_GGUF_PATH}"
LLAMACPP="{JETSON_LLAMACPP}"
SCRIPTS="{JETSON_DEST}/bench_scripts"
OUT="{JETSON_DEST}/results"
mkdir -p "$OUT"

echo
echo "============================================================"
echo "Llama 3.2 1B — Jetson Orin Nano Benchmark Suite"
echo "============================================================"
echo "GGUF:    $GGUF"
echo "llama-cli: $LLAMACPP/llama-cli"
echo

# Verify llama-cli is available
if [ ! -f "$LLAMACPP/llama-cli" ]; then
    echo "ERROR: llama-cli not found at $LLAMACPP/llama-cli"
    echo "Build it: cd /opt/llama.cpp/build && cmake --build . -j4 -t llama-cli"
    exit 1
fi

echo "[1/3] Cognitive Shift Stress Test..."
python3 "$SCRIPTS/bench1_cognitive_shift.py" \\
    --gguf "$GGUF" --llamacpp "$LLAMACPP" \\
    --json-out "$OUT/bench1_cognitive_shift.json"

echo
echo "[2/3] Long-Context Breadcrumb Memory Test..."
python3 "$SCRIPTS/bench2_breadcrumb_memory.py" \\
    --gguf "$GGUF" --llamacpp "$LLAMACPP" \\
    --json-out "$OUT/bench2_breadcrumb_memory.json"

echo
echo "[3/3] Power + Memory Efficiency Reasoning Test..."
python3 "$SCRIPTS/bench3_power_reasoning.py" \\
    --gguf "$GGUF" --llamacpp "$LLAMACPP" \\
    --json-out "$OUT/bench3_power_reasoning.json"

echo
echo "============================================================"
echo "All benchmarks complete."
echo "Results in: $OUT/"
ls -lh "$OUT/"
""")

MASTER_SH.chmod(0o755)
print(f"✓ Written: {MASTER_SH}")
print()
print("Bench scripts:")
for f in sorted((OUTPUT_DIR / "bench_scripts").glob("*")):
    print(f"  {f}")

In [ ]:
# Cell 13 — SCP everything to Jetson
import subprocess, sys

JETSON_SSH = f"{JETSON_USER}@{JETSON_HOST}"
JETSON_SCRIPTS_DEST = f"{JETSON_DEST}/bench_scripts"

print("Creating destination directory on Jetson...")
r = subprocess.run(
    ["ssh", JETSON_SSH, f"mkdir -p {JETSON_DEST} {JETSON_SCRIPTS_DEST}"],
    capture_output=True, text=True,
)
if r.returncode != 0:
    print(f"⚠ SSH mkdir failed: {r.stderr}")
    print("  Is the Jetson reachable? Check JETSON_HOST in Cell 3.")
else:
    print("✓ Remote dirs ready")

    # SCP GGUF
    print(f"\nSCP GGUF ({GGUF_PATH.stat().st_size/1024**3:.2f} GB) — this may take a few minutes...")
    r2 = subprocess.run(
        ["scp", str(GGUF_PATH), f"{JETSON_SSH}:{JETSON_DEST}/"],
        capture_output=False,
    )
    if r2.returncode == 0:
        print(f"✓ GGUF → {JETSON_SSH}:{JETSON_DEST}/{GGUF_PATH.name}")
    else:
        print("✗ GGUF SCP failed")

    # SCP bench scripts
    print("\nSCP benchmark scripts...")
    r3 = subprocess.run(
        ["scp", "-r", str(OUTPUT_DIR / "bench_scripts"),
         f"{JETSON_SSH}:{JETSON_DEST}/"],
        capture_output=False,
    )
    if r3.returncode == 0:
        print(f"✓ Scripts → {JETSON_SSH}:{JETSON_SCRIPTS_DEST}/")
    else:
        print("✗ Script SCP failed")

print()
print("="*62)
print("SSH INTO JETSON AND RUN:")
print("="*62)
print(f"  ssh {JETSON_SSH}")
print(f"  bash {JETSON_DEST}/bench_scripts/run_jetson_bench.sh")
print()
print("Or run individual benchmarks:")
print(f"  # Benchmark 1 — Cognitive Shift")
print(f"  python3 {JETSON_SCRIPTS_DEST}/bench1_cognitive_shift.py \\")
print(f"    --gguf {JETSON_DEST}/{GGUF_PATH.name} \\")
print(f"    --llamacpp {JETSON_LLAMACPP}")
print()
print(f"  # Benchmark 2 — Breadcrumb Memory")
print(f"  python3 {JETSON_SCRIPTS_DEST}/bench2_breadcrumb_memory.py \\")
print(f"    --gguf {JETSON_DEST}/{GGUF_PATH.name} \\")
print(f"    --llamacpp {JETSON_LLAMACPP}")
print()
print(f"  # Benchmark 3 — Power + Reasoning")
print(f"  python3 {JETSON_SCRIPTS_DEST}/bench3_power_reasoning.py \\")
print(f"    --gguf {JETSON_DEST}/{GGUF_PATH.name} \\")
print(f"    --llamacpp {JETSON_LLAMACPP}")
print()
print("Results saved to JSON in: {JETSON_DEST}/results/")

## First-time Jetson Setup

Run this **once** on the Jetson to build llama.cpp for ARM64 + CUDA:

```bash
# Install build deps
sudo apt-get install -y cmake libcurl4-openssl-dev

# Clone and build for Orin (sm_87)
git clone https://github.com/ggerganov/llama.cpp /opt/llama.cpp
cd /opt/llama.cpp
mkdir build && cd build
cmake .. -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=87
cmake --build . -j4 -t llama-cli

# Verify
/opt/llama.cpp/build/bin/llama-cli --version
```

## Interpreting Results

### Benchmark 1 — Cognitive Shift
- **Prefill time** is the time to absorb the 3K story tokens — expect ~2–5s on Orin 1B
- **Tok/s after pivot** shows if quality degrades after context switch — 1B models can struggle
- **Response quality**: does it solve the logic puzzle or regurgitate story fragments?

### Benchmark 2 — Breadcrumb Memory
- Score **5/5 at 2K** is expected — trivial for any model
- Score **3–4/5 at 4K** is typical for 1B — medium memory stress
- Score **<3/5 at 8K** reveals context window degradation — 1B models lose older facts first
- Watch which breadcrumbs are forgotten: the earliest-planted ones (primacy vs recency tradeoff)

### Benchmark 3 — Power + Memory
- **Peak power (total)**: Orin Nano TDP is 10W; LLM inference typically draws 5–8W
- **EMC MHz**: Higher = more memory traffic = harder reasoning → correlates with reasoning depth
- **Watts/token**: `avg_total_mw / 1000 / tok_per_s` = energy per token; lower is more efficient
- Compare Task A (code debug) vs Task B (riddles) to see which type of reasoning costs more power

### Llama 3.2 1B on Orin Nano — Expected ranges

| Metric | Expected (NVMe) | Expected (microSD) |
|--------|-----------------|--------------------|
| Model load time | 0.8–1.5s | 8–15s |
| Tok/s (ngl=99) | 30–55 tok/s | 25–45 tok/s |
| Peak power | 4–7W | 4–7W |
| 2K breadcrumb score | 5/5 | 5/5 |
| 8K breadcrumb score | 1–3/5 | 1–3/5 |